In [ ]:
"""
Stage 5 — Quant Evaluation Suite
Stock Return Prediction | Quant Research Roadmap — Tier 2

Loads trained models from Stage 4, runs predictions on the test set,
and evaluates them the way a quant desk would — NOT just RMSE.

Metrics computed:
  ML metrics    : RMSE, MAE, R²
  Quant metrics : Sharpe ratio, Sortino ratio, Max drawdown,
                  Calmar ratio, Directional accuracy, Hit rate by quintile,
                  Information Coefficient (IC), Annualised return

Outputs:
  results/TICKER_metrics.csv       ← all metrics in one row per ticker
  results/TICKER_equity_curve.csv  ← daily P&L for plotting
  results/summary.csv              ← cross-ticker comparison table

Why these metrics matter more than RMSE:
  A model can have low RMSE but still be useless for trading if it
  gets direction wrong on the big move days. Sharpe ratio, directional
  accuracy, and IC tell you whether the signal is actually tradeable.
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from main import load_sequences, SEQ_DIR
from lstm_model import load_model

RESULTS_DIR   = "results"
MODEL_DIR     = "models"
TICKERS       = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "KOTAKBANK",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "TITAN",
]

TRADING_DAYS  = 252
TX_COST       = 0.001        # 0.1% per trade (realistic NSE estimate)
RISK_FREE     = 0.065        # 6.5% annual (approx Indian 10-yr G-Sec yield)
RF_DAILY      = RISK_FREE / TRADING_DAYS


# ═══════════════════════════════════════════════════════════════════════════════
# ML METRICS
# ═══════════════════════════════════════════════════════════════════════════════

def ml_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Standard regression metrics.
    These are the LEAST important metrics for quant evaluation —
    included for completeness and because interviewers still ask.

    R² < 0 means the model is worse than predicting the mean every day.
    For financial returns, R² of 0.01–0.05 is actually considered useful
    (markets are hard to predict — don't expect 0.9).
    """
    residuals = y_true - y_pred
    ss_res    = (residuals ** 2).sum()
    ss_tot    = ((y_true - y_true.mean()) ** 2).sum()

    return {
        "rmse" : float(np.sqrt(np.mean(residuals ** 2))),
        "mae"  : float(np.mean(np.abs(residuals))),
        "r2"   : float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# DIRECTIONAL ACCURACY
# ═══════════════════════════════════════════════════════════════════════════════

def directional_accuracy(y_true: np.ndarray,
                         y_pred: np.ndarray) -> dict:
    """
    What % of days did the model correctly predict the direction (up/down)?

    50% = random coin flip — no better than guessing.
    55% = genuinely useful for a long/short strategy.
    60%+ = strong signal (rare in liquid markets).

    We also compute:
      - Hit rate on large moves only (|return| > 1%)
        These are the days that actually matter for P&L.
        A model that gets small-move days right but big-move days wrong
        is useless and possibly dangerous.

      - Bull/bear breakdown:
        Does the model predict up days better than down days?
        Useful for diagnosing asymmetric bias.
    """
    true_dir  = (y_true > 0).astype(int)
    pred_dir  = (y_pred > 0).astype(int)
    correct   = (true_dir == pred_dir)

    large_mask = np.abs(y_true) > 0.01
    large_acc  = correct[large_mask].mean() if large_mask.sum() > 0 else np.nan

    up_mask    = y_true > 0
    up_acc     = correct[up_mask].mean() if up_mask.sum() > 0 else np.nan

    down_mask  = y_true < 0
    down_acc   = correct[down_mask].mean() if down_mask.sum() > 0 else np.nan

    return {
        "directional_accuracy"  : float(correct.mean()),
        "large_move_accuracy"   : float(large_acc),
        "up_day_accuracy"       : float(up_acc),
        "down_day_accuracy"     : float(down_acc),
        "n_large_moves"         : int(large_mask.sum()),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEGY SIMULATION
# ═══════════════════════════════════════════════════════════════════════════════

def simulate_strategy(y_true: np.ndarray,
                      y_pred: np.ndarray,
                      dates:  pd.DatetimeIndex,
                      threshold: float = 0.0) -> pd.DataFrame:
    """
    Simulates a simple long/short strategy driven by the model's signal.

    Rules:
      If predicted return > +threshold  → long  (+1)
      If predicted return < -threshold  → short (-1)
      Otherwise                         → flat  (0)

    Transaction cost applied on every position change only.
    A 5-day long position pays tx cost once on entry and once on exit.

    strategy_return[t] = position[t] x actual_return[t] - tx_cost_if_traded
    """
    positions = np.where(y_pred >  threshold,  1,
                np.where(y_pred < -threshold, -1, 0)).astype(float)

    position_changes = np.diff(positions, prepend=0)
    tx_costs         = np.abs(position_changes) * TX_COST

    strategy_ret  = positions * y_true - tx_costs
    buyhold_ret   = y_true.copy()

    df = pd.DataFrame({
        "date"          : dates,
        "actual_return" : y_true,
        "pred_return"   : y_pred,
        "position"      : positions,
        "strategy_ret"  : strategy_ret,
        "buyhold_ret"   : buyhold_ret,
    }).set_index("date")

    df["strategy_equity"] = (1 + df["strategy_ret"]).cumprod()
    df["buyhold_equity"]  = (1 + df["buyhold_ret"]).cumprod()

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# RISK / RETURN METRICS
# ═══════════════════════════════════════════════════════════════════════════════

def sharpe_ratio(returns: np.ndarray,
                 rf_daily: float = RF_DAILY) -> float:
    """
    Sharpe = (mean_daily_excess_return / std_daily_return) x sqrt(252)

    Interpretation:
      < 0    : losing money after risk-free rate
      0–0.5  : weak
      0.5–1  : acceptable
      1–2    : good (most hedge funds target this)
      > 2    : excellent (check carefully for overfitting)
    """
    excess = returns - rf_daily
    if excess.std() == 0:
        return 0.0
    return float((excess.mean() / excess.std()) * np.sqrt(TRADING_DAYS))


def sortino_ratio(returns: np.ndarray,
                  rf_daily: float = RF_DAILY) -> float:
    """
    Like Sharpe but penalises DOWNSIDE volatility only.
    Upside volatility is not a risk — Sharpe penalises it, Sortino doesn't.
    More appropriate for equity strategies with asymmetric return distributions.
    """
    excess   = returns - rf_daily
    downside = excess[excess < 0]
    if len(downside) == 0 or downside.std() == 0:
        return np.nan
    return float((excess.mean() / downside.std()) * np.sqrt(TRADING_DAYS))


def max_drawdown(equity_curve: np.ndarray) -> float:
    """
    Maximum peak-to-trough decline.
    Always negative. -0.25 = portfolio fell 25% from its previous high.

    Critical: most institutional desks hard-limit strategies to MDD < 15-20%.
    A Sharpe of 1.5 with MDD of -60% is usually untradeable in practice.
    """
    rolling_max = np.maximum.accumulate(equity_curve)
    drawdowns   = (equity_curve - rolling_max) / rolling_max
    return float(drawdowns.min())


def calmar_ratio(annualised_return: float, mdd: float) -> float:
    """
    Calmar = annualised_return / |max_drawdown|
    Measures return per unit of worst-case drawdown risk.
    > 1 means you're generating more return than your worst drawdown.
    """
    if mdd == 0:
        return np.nan
    return float(annualised_return / abs(mdd))


# ═══════════════════════════════════════════════════════════════════════════════
# INFORMATION COEFFICIENT
# ═══════════════════════════════════════════════════════════════════════════════

def information_coefficient(y_true: np.ndarray,
                             y_pred: np.ndarray) -> dict:
    """
    IC = Spearman rank correlation between predicted and actual returns.

    Why Spearman not Pearson:
      We care whether the model correctly RANKS days — predicts that
      a high-return day is higher than a low-return day.
      Rank correlation captures this. Linear correlation doesn't.

    Interpretation:
      IC = 0    : no predictive power
      IC = 0.05 : weak but usable (common in live quant funds)
      IC = 0.10 : strong
      IC > 0.15 : exceptional (check for lookahead bias)

    ICIR = mean(rolling IC) / std(rolling IC)
      High ICIR = consistent signal. More valuable than a high but
      volatile IC that averages well but is unreliable day-to-day.
    """
    ic, p_value = stats.spearmanr(y_pred, y_true)

    n = len(y_true)
    rolling_ic = []
    for i in range(21, n):
        window_ic, _ = stats.spearmanr(y_pred[i-21:i], y_true[i-21:i])
        rolling_ic.append(window_ic)

    rolling_ic = np.array(rolling_ic)
    icir = float(rolling_ic.mean() / rolling_ic.std()) if rolling_ic.std() > 0 else np.nan

    return {
        "ic"      : float(ic),
        "ic_pval" : float(p_value),
        "icir"    : float(icir),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# QUINTILE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

def quintile_analysis(y_true: np.ndarray,
                      y_pred: np.ndarray) -> dict:
    """
    Rank predictions into 5 buckets. Check if top quintile outperforms bottom.

    This is how factor researchers validate signals at quant funds.
    A good signal shows a monotonic pattern:
      Q1 (lowest pred) → Q5 (highest pred) returns increase steadily.

    Spread = Q5_mean - Q1_mean
      > 0      : model distinguishes high from low return days
      > 0.001  : ~25% annualised spread — strong alpha
    """
    df = pd.DataFrame({"pred": y_pred, "actual": y_true})
    df["quintile"] = pd.qcut(df["pred"], q=5,
                             labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    qmeans = df.groupby("quintile", observed=True)["actual"].mean()

    result = {f"quintile_{q}_mean_ret": float(qmeans.get(q, np.nan))
              for q in ["Q1", "Q2", "Q3", "Q4", "Q5"]}

    q1 = qmeans.get("Q1", np.nan)
    q5 = qmeans.get("Q5", np.nan)
    result["quintile_spread"] = float(q5 - q1) if not (np.isnan(q1) or np.isnan(q5)) else np.nan
    result["monotonic"]       = bool(qmeans.is_monotonic_increasing)
    return result


# ═══════════════════════════════════════════════════════════════════════════════
# FULL EVALUATION FOR ONE TICKER
# ═══════════════════════════════════════════════════════════════════════════════

def evaluate_ticker(ticker: str) -> dict:
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {ticker}")
    print(f"{'─'*60}")

    X_train, y_train, X_test, y_test, train_dates, test_dates = \
        load_sequences(ticker)

    model      = load_model(ticker, "regression")
    y_pred_raw = model.predict(X_test, verbose=0).flatten()
    y_true     = y_test.flatten()

    print(f"  Test samples : {len(y_true)}")
    print(f"  Test period  : {test_dates.iloc[0].date()} → {test_dates.iloc[-1].date()}")

    # 1. ML metrics
    ml = ml_metrics(y_true, y_pred_raw)
    print(f"\n  ML metrics:")
    print(f"    RMSE : {ml['rmse']:.6f}")
    print(f"    MAE  : {ml['mae']:.6f}")
    print(f"    R²   : {ml['r2']:.4f}  {'⚠ worse than mean' if ml['r2'] < 0 else ''}")

    # 2. Directional accuracy
    da = directional_accuracy(y_true, y_pred_raw)
    print(f"\n  Directional accuracy:")
    print(f"    Overall          : {da['directional_accuracy']:.2%}")
    print(f"    Large moves only : {da['large_move_accuracy']:.2%}  (n={da['n_large_moves']})")
    print(f"    Up days          : {da['up_day_accuracy']:.2%}")
    print(f"    Down days        : {da['down_day_accuracy']:.2%}")

    # 3. Information coefficient
    ic_metrics = information_coefficient(y_true, y_pred_raw)
    print(f"\n  Information Coefficient:")
    print(f"    IC   : {ic_metrics['ic']:.4f}  (p={ic_metrics['ic_pval']:.4f})")
    print(f"    ICIR : {ic_metrics['icir']:.4f}")

    # 4. Quintile analysis
    q_metrics = quintile_analysis(y_true, y_pred_raw)
    print(f"\n  Quintile analysis:")
    for q in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
        print(f"    {q}: {q_metrics[f'quintile_{q}_mean_ret']:+.5f}")
    print(f"    Spread (Q5-Q1) : {q_metrics['quintile_spread']:+.5f}")
    print(f"    Monotonic      : {q_metrics['monotonic']}")

    # 5. Strategy simulation
    equity_df    = simulate_strategy(y_true, y_pred_raw, test_dates)
    strat_rets   = equity_df["strategy_ret"].values
    buyhold_rets = equity_df["buyhold_ret"].values

    n_days        = len(strat_rets)
    strat_ann_ret = float((1 + strat_rets).prod() ** (TRADING_DAYS / n_days) - 1)
    bh_ann_ret    = float((1 + buyhold_rets).prod() ** (TRADING_DAYS / n_days) - 1)

    strat_sharpe  = sharpe_ratio(strat_rets)
    strat_sortino = sortino_ratio(strat_rets)
    strat_mdd     = max_drawdown(equity_df["strategy_equity"].values)
    strat_calmar  = calmar_ratio(strat_ann_ret, strat_mdd)
    bh_sharpe     = sharpe_ratio(buyhold_rets)
    bh_mdd        = max_drawdown(equity_df["buyhold_equity"].values)

    print(f"\n  Strategy vs Buy-and-Hold:")
    print(f"{'':30}{'Strategy':>12}{'Buy & Hold':>12}")
    print(f"  {'Annualised return':<28}{strat_ann_ret:>+11.2%}{bh_ann_ret:>+11.2%}")
    print(f"  {'Sharpe ratio':<28}{strat_sharpe:>12.3f}{bh_sharpe:>12.3f}")
    print(f"  {'Sortino ratio':<28}{strat_sortino:>12.3f}{'—':>12}")
    print(f"  {'Max drawdown':<28}{strat_mdd:>+11.2%}{bh_mdd:>+11.2%}")
    print(f"  {'Calmar ratio':<28}{strat_calmar:>12.3f}{'—':>12}")

    os.makedirs(RESULTS_DIR, exist_ok=True)
    equity_df.to_csv(os.path.join(RESULTS_DIR, f"{ticker}_equity_curve.csv"))

    return {
        "ticker"               : ticker,
        "test_start"           : str(test_dates.iloc[0].date()),
        "test_end"             : str(test_dates.iloc[-1].date()),
        "n_test_days"          : len(y_true),
        **ml,
        **da,
        **ic_metrics,
        **q_metrics,
        "strat_ann_return"     : strat_ann_ret,
        "strat_sharpe"         : strat_sharpe,
        "strat_sortino"        : strat_sortino,
        "strat_max_drawdown"   : strat_mdd,
        "strat_calmar"         : strat_calmar,
        "bh_ann_return"        : bh_ann_ret,
        "bh_sharpe"            : bh_sharpe,
        "bh_max_drawdown"      : bh_mdd,
        "beats_buyhold_sharpe" : int(strat_sharpe > bh_sharpe),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# CROSS-TICKER SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

def print_summary(summary_df: pd.DataFrame):
    ranked = summary_df.sort_values("strat_sharpe", ascending=False)
    print("\n" + "=" * 78)
    print("CROSS-TICKER SUMMARY  (ranked by Strategy Sharpe)")
    print("=" * 78)
    print(f"{'Ticker':<14}{'Dir Acc':>8}{'IC':>7}{'Sharpe':>8}"
          f"{'Sortino':>9}{'MDD':>8}{'Ann Ret':>9}{'> B&H':>6}")
    print("─" * 78)
    for _, row in ranked.iterrows():
        beats = "✓" if row["beats_buyhold_sharpe"] else "✗"
        print(
            f"{row['ticker']:<14}"
            f"{row['directional_accuracy']:>8.2%}"
            f"{row['ic']:>7.4f}"
            f"{row['strat_sharpe']:>8.3f}"
            f"{row['strat_sortino']:>9.3f}"
            f"{row['strat_max_drawdown']:>8.2%}"
            f"{row['strat_ann_return']:>+9.2%}"
            f"{beats:>6}"
        )
    print("─" * 78)
    print(
        f"{'Average':<14}"
        f"{ranked['directional_accuracy'].mean():>8.2%}"
        f"{ranked['ic'].mean():>7.4f}"
        f"{ranked['strat_sharpe'].mean():>8.3f}"
        f"{ranked['strat_sortino'].mean():>9.3f}"
        f"{ranked['strat_max_drawdown'].mean():>8.2%}"
        f"{ranked['strat_ann_return'].mean():>+9.2%}"
    )
    print("=" * 78)


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("Stage 5 — Quant Evaluation Suite")
    print("=" * 60)

    all_results = []

    for ticker in TICKERS:
        model_path = os.path.join(MODEL_DIR, f"{ticker}_regression.keras")
        if not os.path.exists(model_path):
            print(f"\n  SKIP {ticker} — model not found. Run Stage 4 first.")
            continue
        try:
            metrics = evaluate_ticker(ticker)
            all_results.append(metrics)
        except Exception as e:
            print(f"\n  ERROR on {ticker}: {e}")

    if not all_results:
        print("No results. Check model and sequence files.")
        sys.exit(1)

    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary_df = pd.DataFrame(all_results)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
    print(f"\nFull metrics → results/summary.csv")

    print_summary(summary_df)

    print("""
Next: Stage 6 — Backtesting
  Position sizing (Kelly Criterion, vol-scaled)
  Realistic slippage and market impact
  Rolling walk-forward validation

  Load equity curves with:
    eq = pd.read_csv("results/RELIANCE_equity_curve.csv",
                     index_col="date", parse_dates=True)
""")